# SOS Baby Monitor — Training Notebook
**הוראות:**
1. העלי את תיקיית `data/raw` כ-zip (ראי תא 2)
2. הריצי את כל התאים לפי הסדר
3. בסוף תורידי את `sos_model.keras` ו-`norm_stats.json`

In [ ]:
# ===== תא 1: התקנת ספריות =====
!pip install -q librosa soundfile

In [ ]:
# ===== תא 2: העלאת הדאטא המקומי =====
# צרי zip מתיקיית data/raw אצלך במחשב והעלי אותו כאן
# איך ליצור zip: לחצי ימין על תיקיית data/raw -> Send to -> Compressed folder

from google.colab import files
import zipfile, os

print('העלי את קובץ ה-zip של data/raw...')
uploaded = files.upload()  # תבחרי את הקובץ

zip_name = list(uploaded.keys())[0]
print(f'מחלץ {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/data')
os.remove(zip_name)

# בדיקה
for cat in ['crying', 'background']:
    path = f'/content/data/raw/{cat}'
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f'{cat}: {count} files')
    else:
        print(f'WARNING: {path} not found!')

In [ ]:
# ===== תא 3: הורדת FSD50K והוספת רעשי רקע רלוונטיים =====
# מוריד רק את ה-metadata ואז רק הקבצים הרלוונטיים
import os, shutil, csv, urllib.request, zipfile

FSD50K_DIR = '/content/fsd50k_tmp'
BG_DIR     = '/content/data/raw/background'
os.makedirs(FSD50K_DIR, exist_ok=True)

# קטגוריות FSD50K רלוונטיות לרעשי בית עם תינוק
# אלה labels מה-FSD50K ontology שמתאימים לסביבת בית
RELEVANT_LABELS = {
    # דיבור ואנשים
    'Speech', 'Male speech, man speaking', 'Female speech, woman speaking',
    'Conversation', 'Narration, monologue', 'Child speech, kid speaking',
    # מוזיקה
    'Music', 'Musical instrument', 'Singing', 'Song',
    'Lullaby', 'Piano', 'Guitar',
    # טלוויזיה ומדיה
    'Television', 'Radio', 'Telephone',
    # רעשי בית
    'Dishes, pots, and pans', 'Cutlery, silverware',
    'Microwave oven', 'Blender', 'Sink (filling or washing)',
    'Water tap, faucet', 'Toilet flush',
    'Door', 'Knock', 'Slam',
    'Printer', 'Computer keyboard', 'Mouse',
    'Alarm clock', 'Ringtone',
}

# הורדת metadata בלבד (קובץ CSV קטן)
print('מוריד FSD50K metadata...')
!wget -q "https://zenodo.org/record/4060432/files/FSD50K.ground_truth.zip" -O /content/fsd50k_meta.zip
with zipfile.ZipFile('/content/fsd50k_meta.zip', 'r') as z:
    z.extractall('/content/fsd50k_meta')
os.remove('/content/fsd50k_meta.zip')
print('metadata הורד')

In [ ]:
# ===== תא 4: בחירת קבצים רלוונטיים מ-FSD50K והורדתם =====
import pandas as pd

# קריאת ה-metadata
dev_csv  = '/content/fsd50k_meta/FSD50K.ground_truth/dev.csv'
eval_csv = '/content/fsd50k_meta/FSD50K.ground_truth/eval.csv'

def get_relevant_ids(csv_path, relevant_labels, max_per_label=50):
    df = pd.read_csv(csv_path)
    selected = []
    label_counts = {}
    for _, row in df.iterrows():
        labels = str(row['labels']).split(',')
        for label in labels:
            label = label.strip()
            if label in relevant_labels:
                if label_counts.get(label, 0) < max_per_label:
                    selected.append((str(row['fname']), label))
                    label_counts[label] = label_counts.get(label, 0) + 1
                    break
    return selected

dev_files  = get_relevant_ids(dev_csv,  RELEVANT_LABELS, max_per_label=40)
eval_files = get_relevant_ids(eval_csv, RELEVANT_LABELS, max_per_label=20)
all_files  = dev_files + eval_files

print(f'נמצאו {len(all_files)} קבצים רלוונטיים ב-FSD50K')

# הורדת הקבצים עצמם מ-Zenodo
# FSD50K מחולק לחלקים — נוריד רק dev/eval audio
print('מוריד FSD50K dev audio (חלק 1/2)...')
!wget -q "https://zenodo.org/record/4060432/files/FSD50K.dev_audio.z01" -O /content/fsd50k_dev.z01
!wget -q "https://zenodo.org/record/4060432/files/FSD50K.dev_audio.z02" -O /content/fsd50k_dev.z02
!wget -q "https://zenodo.org/record/4060432/files/FSD50K.dev_audio.z03" -O /content/fsd50k_dev.z03
!wget -q "https://zenodo.org/record/4060432/files/FSD50K.dev_audio.z04" -O /content/fsd50k_dev.z04
!wget -q "https://zenodo.org/record/4060432/files/FSD50K.dev_audio.z05" -O /content/fsd50k_dev.z05
!wget -q "https://zenodo.org/record/4060432/files/FSD50K.dev_audio.zip" -O /content/fsd50k_dev.zip
print('מחלץ...')
!zip -s 0 /content/fsd50k_dev.zip --out /content/fsd50k_dev_full.zip
!unzip -q /content/fsd50k_dev_full.zip -d /content/fsd50k_audio
!rm /content/fsd50k_dev*.z* /content/fsd50k_dev_full.zip
print('dev audio מוכן')

In [ ]:
# ===== תא 5: העתקת הקבצים הרלוונטיים ל-background =====
import shutil

audio_base = '/content/fsd50k_audio/FSD50K.dev_audio'
copied = 0
missing = 0

for fname, label in dev_files:
    src = os.path.join(audio_base, f'{fname}.wav')
    dst = os.path.join(BG_DIR, f'fsd50k_{fname}.wav')
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        copied += 1
    elif not os.path.exists(src):
        missing += 1

# ניקוי
shutil.rmtree('/content/fsd50k_audio', ignore_errors=True)
shutil.rmtree('/content/fsd50k_meta',  ignore_errors=True)
shutil.rmtree(FSD50K_DIR,              ignore_errors=True)

print(f'הועתקו: {copied} קבצים מ-FSD50K')
print(f'לא נמצאו: {missing}')
print(f'סהכ background: {len(os.listdir(BG_DIR))}')
print(f'סהכ crying:     {len(os.listdir("/content/data/raw/crying"))}')

In [ ]:
# ===== תא 6: Preprocessing — WAV/3GP -> mel-spectrogram =====
import numpy as np
import librosa
import warnings
warnings.filterwarnings('ignore')

RAW_DIR       = '/content/data/raw'
PROCESSED_DIR = '/content/data/processed'
CATEGORIES    = ['crying', 'background']
SR            = 22050
DURATION      = 2.0

os.makedirs(PROCESSED_DIR, exist_ok=True)

def process_file(file_path):
    audio, sr = librosa.load(file_path, sr=SR, duration=DURATION, mono=True)
    target_len = int(SR * DURATION)
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]
    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=128)
    return librosa.power_to_db(mel, ref=np.max)

SUPPORTED = ('.wav', '.3gp', '.mp3', '.ogg', '.flac')
total = 0
errors = 0

for category in CATEGORIES:
    cat_dir = os.path.join(RAW_DIR, category)
    files   = [f for f in os.listdir(cat_dir) if f.lower().endswith(SUPPORTED)]
    cat_ok  = 0
    for filename in files:
        save_path = os.path.join(PROCESSED_DIR, f'{category}_{os.path.splitext(filename)[0]}.npy')
        if os.path.exists(save_path):  # דילוג על קבצים שכבר עובדו
            cat_ok += 1
            continue
        try:
            mel = process_file(os.path.join(cat_dir, filename))
            np.save(save_path, mel)
            cat_ok += 1
        except Exception:
            errors += 1
    total += cat_ok
    print(f'{category}: {cat_ok} קבצים')

print(f'\nסהכ: {total} | שגיאות: {errors}')

In [ ]:
# ===== תא 7: אימון המודל =====
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

CATEGORIES    = ['crying', 'background']
PROCESSED_DIR = '/content/data/processed'

# טעינה
X, y = [], []
for label, category in enumerate(CATEGORIES):
    files = [f for f in os.listdir(PROCESSED_DIR) if f.startswith(category)]
    for f in files:
        mel = np.load(os.path.join(PROCESSED_DIR, f))
        X.append(mel)
        y.append(label)

X = np.array(X)
print(f'crying: {y.count(0)} | background: {y.count(1)} | shape: {X.shape}')

# נרמול
mean = float(X.mean())
std  = float(X.std())
X    = (X - mean) / (std + 1e-8)
X    = X[..., np.newaxis]
print(f'נרמול: mean={mean:.2f}, std={std:.2f}')

y_cat = keras.utils.to_categorical(y, num_classes=2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y_cat
)

# Augmentation
def augment(X_raw, y_raw):
    aug_X = [X_raw,
             X_raw + np.random.normal(0, 0.01, X_raw.shape),
             np.roll(X_raw, shift=10, axis=2),
             np.flip(X_raw, axis=2),
             X_raw * np.random.uniform(0.8, 1.2)]
    aug_y = [y_raw] * 5
    return np.concatenate(aug_X), np.concatenate(aug_y)

X_train, y_train = augment(X_train, y_train)
print(f'אחרי augmentation: {len(X_train)} דוגמאות')

# מודל
model = keras.Sequential([
    keras.layers.Input(shape=X.shape[1:]),
    keras.layers.Conv2D(32, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),
    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(2, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# class weights
n_cry = np.sum(y_train[:,0])
n_bg  = np.sum(y_train[:,1])
total = len(y_train)
class_weight = {0: total/(2*n_cry), 1: total/(2*n_bg)}
print(f'class weights: crying={class_weight[0]:.2f}, background={class_weight[1]:.2f}')

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=7, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    class_weight=class_weight
)

In [ ]:
# ===== תא 8: הערכה ו-Confusion Matrix =====
loss, acc = model.evaluate(X_test, y_test)
print(f'דיוק: {acc:.2%}')

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test.argmax(axis=1), y_pred.argmax(axis=1))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CATEGORIES)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Baby Monitor')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png')
plt.show()

# גרף אימון
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'],     label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.legend()
plt.savefig('/content/training_history.png')
plt.show()

In [ ]:
# ===== תא 9: שמירה והורדה =====
import json
from google.colab import files

model.save('/content/sos_model.keras')
with open('/content/norm_stats.json', 'w') as f:
    json.dump({'mean': mean, 'std': std}, f)

print('מוריד קבצים...')
files.download('/content/sos_model.keras')
files.download('/content/norm_stats.json')
files.download('/content/confusion_matrix.png')
files.download('/content/training_history.png')
print('סיום! שמרי את הקבצים ב-src/')